In [ ]:
from pathlib import Path
EXPECTED_GIT_COMMIT = "8efda18ed796fdb6208e1441fece46ff1f343b97"
assert (len(EXPECTED_GIT_COMMIT) == 40 and EXPECTED_GIT_COMMIT != "REPLACE_AFTER_PUSH"), "Pin the reviewed pushed commit before Colab formal training"
REPO_URL = "https://github.com/sfczaa/ddpm-derm-augmentation.git"
RUN_VERSION = "v1_panderm_base_c1_finetune"
# Account label for operational records; excluded from experiment identity.
MANUAL_TAKEOVER_CONFIRMED = False
assert isinstance(MANUAL_TAKEOVER_CONFIRMED, bool), "MANUAL_TAKEOVER_CONFIRMED must be a bool"
# Empty unless a human reviewed the diff between that commit and this one
# and decided it changed orchestration only: no model, data, optimizer,
# hyperparameter, split or checkpoint-source change. Artifacts still stamped
# with it are then accepted even though git_commit is an immutable identity
# field; every other identity field must still match exactly. Never a default.
COMMIT_CARRY_FORWARD_FROM = ""
assert COMMIT_CARRY_FORWARD_FROM == "" or (len(COMMIT_CARRY_FORWARD_FROM) == 40 and all(character in "0123456789abcdef" for character in COMMIT_CARRY_FORWARD_FROM)), "COMMIT_CARRY_FORWARD_FROM must be empty or a full lowercase commit SHA"
assert COMMIT_CARRY_FORWARD_FROM != EXPECTED_GIT_COMMIT, "carry-forward from the pinned commit itself authorizes nothing"
ACCOUNT_LABEL = "A"
assert ACCOUNT_LABEL in {"A", "B", "C"}, "ACCOUNT_LABEL must be A, B, or C"
# Opening and running this notebook is the explicit confirmation to unlock formal training.
FORMAL_TRAINING_CONFIRMED = True
assert isinstance(FORMAL_TRAINING_CONFIRMED, bool), "FORMAL_TRAINING_CONFIRMED must be a bool"
ARCH = "panderm_base_vit_b16"
CHECKPOINT_FORMAT = "panderm_full_model_v1"
VARIANT = "C1"
DF_TARGET_COUNT = 585
FORMAL_EPOCHS = 50
WARMUP_EPOCHS = 10
SEEDS = (0, 1, 2)
BATCH_SIZE = 16
ACCUMULATION_STEPS = 8
EFFECTIVE_BATCH_SIZE = 128
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 0.05
LAYER_DECAY = 0.65
DROP_PATH = 0.2
PANDERM_UPSTREAM_REPO = "https://github.com/SiyuanYan1/PanDerm"
PANDERM_UPSTREAM_COMMIT = "fd7a80748ba7fc3e203fed88f909f4689d0d6f24"
PANDERM_CHECKPOINT_FILENAME = "panderm_bb_data6_checkpoint-499.pth"
PANDERM_CHECKPOINT_DRIVE_ID = "removed-from-public-history"
SHARED_PROJECT_DIR = Path("/content/drive/MyDrive/ddpm-derm-augmentation")
SHARED_RUN_ROOT = Path("/content/drive/MyDrive/ddpm-derm-panderm-runs")
COCA_RUN_ROOT = Path("/content/drive/MyDrive/ddpm-derm-coca-runs")
V1_ROOT = SHARED_RUN_ROOT / RUN_VERSION
VALIDATION_RECORD = V1_ROOT / "validation_record.json"
FORMAL_ROOT = V1_ROOT / "formal"
SHARED_ROOT_SENTINEL = SHARED_RUN_ROOT / ".panderm_shared_root.json"
DATA_CACHE_DIRECTORY = SHARED_RUN_ROOT / "data_cache" / "ham10000_train_val_only_v1"
CODE_DIR = Path("/content/panderm-code")
UPSTREAM_DIR = Path("/content/panderm-upstream")
WEIGHTS_CACHE = Path("/content/panderm-weights")
LOCAL_DATA_DIR = Path("/content/ham10000-data")


# PanDerm-Base C1 full fine-tuning formal training

Prerequisites: both shared Drive shortcuts must resolve physically to the reviewed roots, the user must have Editor permission, and the Colab Secret GH_TOKEN must be enabled for this notebook.

- Project shortcut: `/content/drive/MyDrive/ddpm-derm-augmentation`
- Durable run shortcut: `/content/drive/MyDrive/ddpm-derm-panderm-runs`
- A, B and C run only in sequence. The previous runtime must be stopped before another account resumes the same fixed run version.

50-epoch formal training across seeds {0, 1, 2}. Evaluation scope remains validation_only with test_metrics=None permanently. Test split access is strictly prohibited.


## Phase 0 CHECK - Drive, roots, sentinel, pinned clones, dependencies, checkpoint SHA, isolation


In [ ]:
import base64, hashlib, json, os, re, shutil, subprocess, sys, time, uuid
from google.colab import auth, drive, userdata
drive.mount("/content/drive")
assert SHARED_PROJECT_DIR.is_dir(), f"missing shared project shortcut: {SHARED_PROJECT_DIR}"
assert SHARED_RUN_ROOT.is_dir(), f"missing shared run shortcut: {SHARED_RUN_ROOT}"
def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""): digest.update(chunk)
    return digest.hexdigest()
token = userdata.get("GH_TOKEN")
assert token and len(token) > 20, "Colab Secret GH_TOKEN with read access to this repo is required"
GH_TOKEN_PRESENT = True
assert not CODE_DIR.exists(), f"fresh runtime required: {CODE_DIR}"
basic_credential = base64.b64encode(("x-access-token:" + token).encode()).decode()
clone_env = os.environ.copy()
clone_env["GIT_CONFIG_COUNT"] = "1"
clone_env["GIT_CONFIG_KEY_0"] = "http.https://github.com/.extraheader"
clone_env["GIT_CONFIG_VALUE_0"] = "Authorization: Basic " + basic_credential
try:
    subprocess.run(["git", "clone", REPO_URL, str(CODE_DIR)], check=True, env=clone_env)
finally:
    clone_env["GIT_CONFIG_VALUE_0"] = ""
    token = basic_credential = None
    del token, basic_credential, clone_env
subprocess.run(["git", "-C", str(CODE_DIR), "checkout", "--detach", EXPECTED_GIT_COMMIT], check=True)
commit = subprocess.check_output(["git", "-C", str(CODE_DIR), "rev-parse", "HEAD"], text=True).strip()
remote = subprocess.check_output(["git", "-C", str(CODE_DIR), "remote", "get-url", "origin"], text=True).strip()
status = subprocess.check_output(["git", "-C", str(CODE_DIR), "status", "--short"], text=True).strip()
assert commit == EXPECTED_GIT_COMMIT and not status, "clone must be a clean detached checkout of the pinned commit"
assert "@" not in remote and "x-access-token" not in remote, "clone URL must not embed a credential"
assert not UPSTREAM_DIR.exists(), f"fresh runtime required: {UPSTREAM_DIR}"
subprocess.run(["git", "clone", "--filter=blob:none", PANDERM_UPSTREAM_REPO, str(UPSTREAM_DIR)], check=True)
subprocess.run(["git", "-C", str(UPSTREAM_DIR), "checkout", "--detach", PANDERM_UPSTREAM_COMMIT], check=True)
upstream_commit = subprocess.check_output(["git", "-C", str(UPSTREAM_DIR), "rev-parse", "HEAD"], text=True).strip()
upstream_status = subprocess.check_output(["git", "-C", str(UPSTREAM_DIR), "status", "--short"], text=True).strip()
assert upstream_commit == PANDERM_UPSTREAM_COMMIT and not upstream_status, "PanDerm upstream must be a clean detached checkout"
os.environ["HF_HOME"] = "/content/hf-cache"
os.environ["TORCH_HOME"] = "/content/torch-cache"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm==0.9.16", "gdown>=5.1", "pandas>=2.0", "pillow>=9.0"], check=True)
sys.path.insert(0, str(CODE_DIR / "src"))
from importlib.metadata import version
import torch
from ddpm_derm import panderm_run
assert version("timm") == "0.9.16"
assert panderm_run.UPSTREAM_COMMIT == PANDERM_UPSTREAM_COMMIT == upstream_commit
assert panderm_run.RUN_VERSION == RUN_VERSION and panderm_run.ARCH == ARCH
resolved_root = panderm_run.require_existing_shared_root(SHARED_RUN_ROOT)
SHARED_RUN_ROOT = resolved_root
V1_ROOT = SHARED_RUN_ROOT / RUN_VERSION
VALIDATION_RECORD = V1_ROOT / "validation_record.json"
FORMAL_ROOT = V1_ROOT / "formal"
SHARED_ROOT_SENTINEL = SHARED_RUN_ROOT / ".panderm_shared_root.json"
DATA_CACHE_DIRECTORY = SHARED_RUN_ROOT / "data_cache" / "ham10000_train_val_only_v1"
auth.authenticate_user()
import google.auth
from googleapiclient.discovery import build
drive_credentials, _ = google.auth.default()
DRIVE_API = build("drive", "v3", credentials=drive_credentials, cache_discovery=False)
DRIVE_PROVIDER_FIELDS = "nextPageToken,incompleteSearch,files(id,name,mimeType,parents,driveId,ownedByMe,owners(me,permissionId),resourceKey,trashed,shortcutDetails(targetId,targetMimeType,targetResourceKey))"
def drive_api_query_literal(value):
    return str(value).replace("\\", "\\\\").replace("'", "\\'")
def drive_api_execute(request, resource_keys=()):
    if resource_keys:
        request.headers["X-Goog-Drive-Resource-Keys"] = panderm_run.drive_resource_key_header(tuple(resource_keys))
    return request.execute()
DRIVE_API_ABOUT = drive_api_execute(DRIVE_API.about().get(fields="user(me,permissionId)"))
DRIVE_MY_DRIVE_ROOT_ID = drive_api_execute(DRIVE_API.files().get(fileId="root", fields="id"))["id"]
assert DRIVE_MY_DRIVE_ROOT_ID and DRIVE_MY_DRIVE_ROOT_ID != "root", "the Drive API must report a resolved My Drive root folder id"
def drive_api_list_children(parent_id, name, mime_type, *, drive_id=None, parent_resource_key=None):
    query = " and ".join((f"'{drive_api_query_literal(parent_id)}' in parents", f"name = '{drive_api_query_literal(name)}'", f"mimeType = '{drive_api_query_literal(mime_type)}'", "trashed = false"))
    def fetch_page(page_token):
        kwargs = panderm_run.drive_provider_list_request_kwargs(query=query, fields=DRIVE_PROVIDER_FIELDS, drive_id=drive_id, page_token=page_token)
        request = DRIVE_API.files().list(**kwargs)
        keys = ((parent_id, parent_resource_key),) if parent_resource_key else ()
        return drive_api_execute(request, keys)
    return panderm_run.collect_drive_provider_pages(fetch_page)
shortcut_records = drive_api_list_children("root", SHARED_RUN_ROOT.name, panderm_run.DRIVE_SHORTCUT_MIME_TYPE)
root_folder_records = drive_api_list_children("root", SHARED_RUN_ROOT.name, panderm_run.DRIVE_FOLDER_MIME_TYPE)
DRIVE_SHARED_ROOT_TARGET = panderm_run.require_drive_shared_root_target(shortcut_records, root_folder_records, expected_alias=SHARED_RUN_ROOT.name)
DRIVE_ROOT_ID = DRIVE_SHARED_ROOT_TARGET["target_id"]
DRIVE_ROOT_RESOURCE_KEY = DRIVE_SHARED_ROOT_TARGET["target_resource_key"]
root_request = DRIVE_API.files().get(fileId=DRIVE_ROOT_ID, fields="id,name,mimeType,parents,driveId,ownedByMe,resourceKey,trashed", supportsAllDrives=True)
root_metadata = drive_api_execute(root_request, ((DRIVE_ROOT_ID, DRIVE_ROOT_RESOURCE_KEY),) if DRIVE_ROOT_RESOURCE_KEY else ())
ROOT_DRIVE_ID = root_metadata.get("driveId") or None
def wait_for_one_provider_child(parent_id, name, mime_type, phase, *, drive_id, parent_resource_key):
    started = time.monotonic()
    while True:
        records = drive_api_list_children(parent_id, name, mime_type, drive_id=drive_id, parent_resource_key=parent_resource_key)
        if len(records) == 1: return records[0]
        if len(records) > 1: raise ValueError(f"{phase}: ambiguous duplicate provider children: {records}")
        elapsed = time.monotonic() - started
        print(f"[drive-provider] {phase} visibility wait elapsed={elapsed:.1f}s", flush=True)
        if elapsed >= 30.0: raise FileNotFoundError(f"{phase}: provider child was not visible after 30 seconds")
        time.sleep(5.0)
api_mount_probe_name = f".panderm_api_mount_identity_{uuid.uuid4().hex}.json"
api_mount_probe_path = Path("/content/drive/MyDrive") / api_mount_probe_name
try:
    api_mount_probe_path.write_text('{"probe":"drive-api-fuse-account"}\n', encoding="utf-8")
    api_mount_probe_metadata = wait_for_one_provider_child("root", api_mount_probe_name, panderm_run.DRIVE_JSON_MIME_TYPE, "Phase 0 Drive API/FUSE account probe", drive_id=None, parent_resource_key=None)
    DRIVE_API_FUSE_ACCOUNT_ALIGNMENT = panderm_run.require_drive_api_fuse_account_alignment(api_mount_probe_metadata, expected_probe_name=api_mount_probe_name, expected_root_id=DRIVE_MY_DRIVE_ROOT_ID)
finally:
    api_mount_probe_path.unlink(missing_ok=True)
topology_probe_name = f".panderm_lock_topology_probe_{uuid.uuid4().hex}.json"
topology_probe_path = SHARED_RUN_ROOT / topology_probe_name
try:
    topology_probe_path.write_text('{"probe":"validation-lock-topology"}\n', encoding="utf-8")
    topology_probe_metadata = wait_for_one_provider_child(DRIVE_ROOT_ID, topology_probe_name, panderm_run.DRIVE_JSON_MIME_TYPE, "Phase 0 lock topology probe", drive_id=ROOT_DRIVE_ID, parent_resource_key=DRIVE_ROOT_RESOURCE_KEY)
    VALIDATION_LOCK_STORAGE_TOPOLOGY = panderm_run.require_validation_lock_storage_topology(root_metadata, topology_probe_metadata, expected_root_id=DRIVE_ROOT_ID, expected_probe_name=topology_probe_name)
finally:
    topology_probe_path.unlink(missing_ok=True)
phase0_version_records = drive_api_list_children(DRIVE_ROOT_ID, RUN_VERSION, panderm_run.DRIVE_FOLDER_MIME_TYPE, drive_id=ROOT_DRIVE_ID, parent_resource_key=DRIVE_ROOT_RESOURCE_KEY)
PHASE0_VERSION_PROVIDER = panderm_run.require_validation_version_provider_state(phase0_version_records, expected_parent_id=DRIVE_ROOT_ID, run_version=RUN_VERSION, topology=VALIDATION_LOCK_STORAGE_TOPOLOGY, local_version_exists=V1_ROOT.exists())
if PHASE0_VERSION_PROVIDER is not None:
    phase0_version_resource_key = PHASE0_VERSION_PROVIDER.get("resourceKey")
    phase0_version_resource_key = phase0_version_resource_key or ""
    phase0_active_records = drive_api_list_children(PHASE0_VERSION_PROVIDER["id"], panderm_run.ACTIVE_SESSION_FILENAME, panderm_run.DRIVE_JSON_MIME_TYPE, drive_id=ROOT_DRIVE_ID, parent_resource_key=phase0_version_resource_key)
    if phase0_active_records:
        panderm_run.require_active_session_provider_state(phase0_active_records, expected_parent_id=PHASE0_VERSION_PROVIDER["id"], topology=VALIDATION_LOCK_STORAGE_TOPOLOGY, expected_present=True)
        assert MANUAL_TAKEOVER_CONFIRMED is True, "active session exists; confirm the previous runtime is stopped and use manual takeover"
    else:
        panderm_run.require_active_session_provider_state([], expected_parent_id=PHASE0_VERSION_PROVIDER["id"], topology=VALIDATION_LOCK_STORAGE_TOPOLOGY, expected_present=False)
drive_probe = panderm_run.probe_shared_drive(SHARED_RUN_ROOT)
assert SHARED_ROOT_SENTINEL.is_file(), f"missing shared-root sentinel: {SHARED_ROOT_SENTINEL}"
sentinel = panderm_run.create_or_validate_sentinel(SHARED_ROOT_SENTINEL, shortcut_alias="ddpm-derm-panderm-runs", resolved_path=str(resolved_root), drive_folder_id=DRIVE_ROOT_ID, run_version=RUN_VERSION)
assert sentinel.get("drive_folder_id") in (None, DRIVE_ROOT_ID), "shared-root sentinel Drive folder id drift"
panderm_run.require_shared_root_sentinel_identity(sentinel, shortcut_alias="ddpm-derm-panderm-runs", resolved_root=resolved_root, run_version=RUN_VERSION)
DURABLE_ROOT_PROVIDER_IDENTITY = panderm_run.build_durable_root_provider_identity(root_metadata, expected_root_id=DRIVE_ROOT_ID, shortcut_target_resource_key=DRIVE_ROOT_RESOURCE_KEY, shared_root_uuid=sentinel["shared_root_uuid"], topology=VALIDATION_LOCK_STORAGE_TOPOLOGY)
assert SHARED_RUN_ROOT != COCA_RUN_ROOT and COCA_RUN_ROOT not in SHARED_RUN_ROOT.parents, "PanDerm output root must be isolated from CoCa"
panderm_run.require_no_deployment_contamination(CODE_DIR)
assert VALIDATION_RECORD.exists(), "formal training requires a completed validation record (validation_record.json missing)"
assert not (FORMAL_ROOT / "formal_record.json").exists(), "existing formal record blocks a fresh run; inspect manually"
subprocess.run(["nvidia-smi"], check=True)
assert torch.cuda.is_available(), "CUDA is required for formal training"
print(json.dumps({"commit": commit, "upstream_commit": upstream_commit, "drive_probe": drive_probe, "shared_root_source": DRIVE_SHARED_ROOT_TARGET["source"], "storage_topology": VALIDATION_LOCK_STORAGE_TOPOLOGY, "manual_takeover_confirmed": MANUAL_TAKEOVER_CONFIRMED, "formal_training_confirmed": FORMAL_TRAINING_CONFIRMED}, indent=2))


### Phase 0 CHECK - official checkpoint download and pinned SHA-256


In [ ]:
WEIGHTS_CACHE.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = WEIGHTS_CACHE / PANDERM_CHECKPOINT_FILENAME
if not CHECKPOINT_PATH.is_file():
    import gdown
    gdown.download(id=PANDERM_CHECKPOINT_DRIVE_ID, output=str(CHECKPOINT_PATH), quiet=False)
assert CHECKPOINT_PATH.is_file(), f"PanDerm checkpoint download failed: {CHECKPOINT_PATH}"
observed_checkpoint_sha256 = sha256(CHECKPOINT_PATH)
print("checkpoint bytes:", CHECKPOINT_PATH.stat().st_size)
print("checkpoint sha256:", observed_checkpoint_sha256)
print("expected (pinned):", panderm_run.EXPECTED_CHECKPOINT_SHA256)
checkpoint_sha256 = panderm_run.require_checkpoint_sha256(CHECKPOINT_PATH)
provenance_clearance = panderm_run.require_provenance_clearance(
    upstream_commit=upstream_commit,
    checkpoint_sha256=checkpoint_sha256,
    purpose=panderm_run.FORMAL_TRAINING,
    formal_training_confirmed=FORMAL_TRAINING_CONFIRMED,
)
assert provenance_clearance["formal_training_allowed"] is True
assert provenance_clearance["test_access_allowed"] is False
assert provenance_clearance["cleared_for"] == panderm_run.FORMAL_TRAINING
assert provenance_clearance["claim_boundary"] == "suggestive_exploratory_only"
print(json.dumps(provenance_clearance, indent=2))


## Phase 1 CHECK - checkpoint layout, real CPU model load, dependency versions


In [ ]:
import gc
assert "DDPM_DERM_DATA_DIR" not in os.environ, "Phase 1 must not bind a training dataset"
from ddpm_derm import panderm
phase1_started = time.monotonic()
print("[Phase 1] START checkpoint/model/identity checks", flush=True)
phase1_factory = panderm.load_upstream_model_factory(UPSTREAM_DIR)
pretrained_state = panderm.load_pretrained_state(CHECKPOINT_PATH)
pretrained_layout = panderm.detect_checkpoint_layout(pretrained_state)
assert pretrained_layout == panderm.LAYOUT_DIRECT_BACKBONE
remapped_state = panderm.remap_pretrained_state_dict(pretrained_state, layout=pretrained_layout)
assert "fc_norm.weight" in remapped_state and "fc_norm.bias" in remapped_state
assert not any(key.startswith(("norm.", "head.")) for key in remapped_state)
train_transform = panderm.build_train_transform()
eval_transform = panderm.build_eval_transform()
preflight_model = panderm.build_panderm_classifier(checkpoint_path=CHECKPOINT_PATH, upstream_dir=UPSTREAM_DIR, drop_path=DROP_PATH)
assert preflight_model.pretrained_state_layout == panderm.LAYOUT_DIRECT_BACKBONE
assert preflight_model.head.out_features == 7 and preflight_model.head.in_features == 768
model_details = panderm.model_identity(preflight_model, train_transform=train_transform, eval_transform=eval_transform, checkpoint_sha256=checkpoint_sha256)
dependency_versions = panderm.dependency_versions()
assert type(dependency_versions["torch"]) is str
manifest_sha256 = {split: sha256(SHARED_PROJECT_DIR / "data" / "manifests" / f"{split}.csv") for split in ("train", "val")}
class_mapping_sha256 = sha256(SHARED_PROJECT_DIR / "data" / "manifests" / "class_to_idx.json")
fixed_split_identity = manifest_sha256["train"]
formal_output_identity = f"{sentinel['shared_root_uuid']}:{RUN_VERSION}:formal"
print(json.dumps({"factory": phase1_factory.__name__, "layout": pretrained_layout, "dependency_versions": dependency_versions, "formal_output_identity": formal_output_identity}, indent=2))
del pretrained_state, remapped_state, preflight_model
gc.collect()
print(f"[Phase 1] COMPLETE elapsed={time.monotonic() - phase1_started:.1f}s", flush=True)


## Phase 2 CHECK - targeted test suite execution


In [ ]:
import queue, threading
def run_stream(command, cwd=CODE_DIR, expect_success=True, process_env=None):
    started = time.monotonic()
    process = subprocess.Popen(command, cwd=cwd, env=process_env or test_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    output_queue = queue.Queue()
    def pump_output():
        for line in process.stdout: output_queue.put(line)
        output_queue.put(None)
    threading.Thread(target=pump_output, daemon=True).start()
    while True:
        try: line = output_queue.get(timeout=60)
        except queue.Empty:
            print(f"[subprocess] heartbeat elapsed={time.monotonic() - started:.1f}s pid={process.pid} alive={process.poll() is None}", flush=True)
            continue
        if line is None: break
        lines.append(line); print(line, end="", flush=True)
    code_returned = process.wait()
    if expect_success and code_returned: raise subprocess.CalledProcessError(code_returned, command)
    if not expect_success and not code_returned: raise AssertionError("command unexpectedly succeeded")
    return time.monotonic() - started, "".join(lines)
test_env = os.environ.copy()
test_env["DDPM_DERM_DATA_DIR"] = str(SHARED_PROJECT_DIR / "data")
test_env["PYTHONPATH"] = str(CODE_DIR / "src")
test_env["PYTHONUNBUFFERED"] = "1"
test_env["PYTHONDONTWRITEBYTECODE"] = "1"
_, targeted_output = run_stream([sys.executable, "-B", "-u", "-m", "unittest", "-v", "tests.test_panderm_blockers", "tests.test_panderm_base_c1_finetune", "tests.test_panderm_notebooks", "tests.test_panderm_fresh_runtime"])
assert "OK" in targeted_output, "targeted tests failed"


## Phase 3 RUN - verified immutable train/val-only archive staging


In [ ]:
os.environ["DDPM_DERM_DATA_DIR"] = str(SHARED_PROJECT_DIR / "data")
from ddpm_derm import train_panderm
FORMAL_DIR = panderm_run.ensure_tree(SHARED_RUN_ROOT, FORMAL_ROOT.relative_to(SHARED_RUN_ROOT))
ACTIVE_SESSION_PATH = panderm_run.active_session_path(SHARED_RUN_ROOT, run_version=RUN_VERSION)
SESSION_HISTORY_DIR = panderm_run.ensure_tree(SHARED_RUN_ROOT, (V1_ROOT / panderm_run.SESSION_HISTORY_DIRECTORY).relative_to(SHARED_RUN_ROOT))
FORMAL_SESSION = panderm_run.session_marker(ACCOUNT_LABEL, "resume", {"run_version": RUN_VERSION, "mode": "formal"})
FORMAL_SESSION_ID = FORMAL_SESSION["session_id"]
base_run_identity = panderm_run.build_run_identity(
    git_commit=commit, seed=0, epochs=FORMAL_EPOCHS, evaluation_scope="validation_only",
    checkpoint_sha256=checkpoint_sha256, model_identity=model_details,
    manifest_sha256=manifest_sha256, fixed_split_identity=fixed_split_identity,
    shared_root_uuid=sentinel["shared_root_uuid"], formal_output_identity=formal_output_identity,
    dependency_versions=dependency_versions, warmup_epochs=WARMUP_EPOCHS, drop_path=DROP_PATH,
    amp_requested=True, amp_effective=True, device_type="cuda",
)
RUN_IDENTITY_SHA256 = panderm_run.session_scope_identity_sha256(base_run_identity)
ACTIVE_SESSION = panderm_run.start_sequential_session(
    ACTIVE_SESSION_PATH, session_id=FORMAL_SESSION_ID, run_version=RUN_VERSION, git_commit=commit,
    shared_root_uuid=sentinel["shared_root_uuid"], account_label=ACCOUNT_LABEL,
    run_identity_sha256=RUN_IDENTITY_SHA256, manual_takeover_confirmed=MANUAL_TAKEOVER_CONFIRMED,
    history_directory=SESSION_HISTORY_DIR,
)
FORMAL_SESSION_ID = ACTIVE_SESSION["session_id"]
if COMMIT_CARRY_FORWARD_FROM:
    COMMIT_CARRY_FORWARD_AUDIT = panderm_run.publish_commit_carry_forward_audit(SESSION_HISTORY_DIR, carried_forward_commit=COMMIT_CARRY_FORWARD_FROM, current_commit=commit, run_version=RUN_VERSION, shared_root_uuid=sentinel["shared_root_uuid"])
    print(json.dumps(COMMIT_CARRY_FORWARD_AUDIT, indent=2))
SESSION_RUNTIME_ENV = {
    "PANDERM_ACTIVE_SESSION_PATH": str(ACTIVE_SESSION_PATH),
    "PANDERM_ACTIVE_SESSION_ID": FORMAL_SESSION_ID,
    "PANDERM_ACTIVE_RUN_VERSION": RUN_VERSION,
    "PANDERM_ACTIVE_GIT_COMMIT": commit,
    "PANDERM_ACTIVE_SHARED_ROOT_UUID": sentinel["shared_root_uuid"],
    "PANDERM_ACTIVE_RUN_IDENTITY_SHA256": RUN_IDENTITY_SHA256,
}
SESSION_GUARD = panderm_run.SequentialSessionWriteGuard.from_environment(SESSION_RUNTIME_ENV)
def require_active_session(phase):
    return SESSION_GUARD.require(phase)
training_env = os.environ.copy()
training_env.update(SESSION_RUNTIME_ENV)
training_env["DDPM_DERM_DATA_DIR"] = str(LOCAL_DATA_DIR)
training_env["PYTHONPATH"] = str(CODE_DIR / "src")
training_env["PYTHONUNBUFFERED"] = "1"
training_env["PYTHONDONTWRITEBYTECODE"] = "1"
APPROVED_CONTENT_IDENTITY = panderm_run.require_approved_content_identity(
    panderm_run.EXPECTED_VALIDATION_CONTENT_IDENTITY_SHA256
)
assert APPROVED_CONTENT_IDENTITY != panderm_run.VALIDATION_CONTENT_IDENTITY_PLACEHOLDER
assert len(APPROVED_CONTENT_IDENTITY) == 64
staging_report = panderm_run.reuse_validation_archive_cache(
    DATA_CACHE_DIRECTORY, Path("/content"), LOCAL_DATA_DIR,
    expected_file_content_identity_sha256=APPROVED_CONTENT_IDENTITY,
    expected_fixed_split_identity=fixed_split_identity,
    expected_manifest_sha256=manifest_sha256,
    expected_class_mapping_sha256=class_mapping_sha256,
    write_guard=require_active_session,
)
assert staging_report["images_staged"] == 8505
assert staging_report["test_manifest_present"] is False
for required_manifest in ("train.csv", "val.csv", "class_to_idx.json"):
    assert (LOCAL_DATA_DIR / "manifests" / required_manifest).is_file()
assert not (LOCAL_DATA_DIR / "manifests" / "test.csv").exists()
print(json.dumps(staging_report, indent=2))


## Phase 4 RUN - 3-seed formal C1 training loop


In [ ]:
formal_runs = []
for seed in SEEDS:
    print(f"[formal-loop] START seed={seed} epochs={FORMAL_EPOCHS}", flush=True)
    run_id = panderm_run.build_run_identity(
        git_commit=commit, seed=seed, epochs=FORMAL_EPOCHS, evaluation_scope="validation_only",
        checkpoint_sha256=checkpoint_sha256, model_identity=model_details,
        manifest_sha256=manifest_sha256, fixed_split_identity=fixed_split_identity,
        shared_root_uuid=sentinel["shared_root_uuid"], formal_output_identity=formal_output_identity,
        dependency_versions=dependency_versions, warmup_epochs=WARMUP_EPOCHS, drop_path=DROP_PATH,
        amp_requested=True, amp_effective=True, device_type="cuda",
    )
    seed_ckpt_dir = FORMAL_ROOT / "checkpoints" / ARCH / f"{VARIANT}_seed{seed}"
    seed_result_path = FORMAL_ROOT / "results" / ARCH / f"results_{VARIANT}_seed{seed}.json"
    cmd = [
        sys.executable, "-B", "-u", "-m", "ddpm_derm.train_panderm",
        "--variant", VARIANT, "--seed", str(seed), "--epochs", str(FORMAL_EPOCHS),
        "--batch-size", str(BATCH_SIZE), "--accumulation-steps", str(ACCUMULATION_STEPS),
        "--lr", str(LEARNING_RATE), "--weight-decay", str(WEIGHT_DECAY),
        "--warmup-epochs", str(WARMUP_EPOCHS), "--layer-decay", str(LAYER_DECAY),
        "--df-target-count", str(DF_TARGET_COUNT), "--checkpoint", str(CHECKPOINT_PATH),
        "--checkpoint-sha256", checkpoint_sha256, "--upstream-dir", str(UPSTREAM_DIR),
        "--upstream-commit", upstream_commit, "--evaluation-scope", "validation_only",
        "--output-dir", str(FORMAL_ROOT), "--run-version", RUN_VERSION,
        "--shared-root-uuid", sentinel["shared_root_uuid"],
        "--formal-output-identity", formal_output_identity,
        "--fixed-split-identity", fixed_split_identity, "--resume",
        "--authorized-commit-carry-forward", COMMIT_CARRY_FORWARD_FROM,
    ]
    run_stream(cmd, process_env=training_env)
    assert seed_result_path.is_file(), f"missing seed result: {seed_result_path}"
    seed_result = json.loads(seed_result_path.read_text(encoding="utf-8"))
    best_pt = seed_ckpt_dir / seed_result["checkpoint_pointer"]["best"]
    last_pt = seed_ckpt_dir / seed_result["checkpoint_pointer"]["last"]
    verified_best, verified_last = train_panderm.load_completed_checkpoint_pair_safe(
        best_path=best_pt, last_path=last_pt, result=seed_result, model=None, expected_identity=run_id, map_location="cpu", authorized_commit_carry_forward=COMMIT_CARRY_FORWARD_FROM
    )
    panderm_run.require_completed_artifact_identities(expected=run_id, result=seed_result, best_checkpoint=verified_best, last_checkpoint=verified_last, authorized_commit_carry_forward=COMMIT_CARRY_FORWARD_FROM)
    formal_runs.append(seed_result)
    print(f"[formal-loop] COMPLETE seed={seed} best_val_df_f1={seed_result['best_val_df_f1']:.4f}", flush=True)


## Phase 5 COMPLETE - formal aggregation and session completion


In [ ]:
from ddpm_derm import config
import numpy as np
aggregate_output = panderm_run.aggregate_results(formal_runs, authorized_commit_carry_forward=COMMIT_CARRY_FORWARD_FROM)
panderm_run.write_json_atomic(
    FORMAL_ROOT / "aggregate_record.json", aggregate_output, write_guard=require_active_session
)
non_collapse_checks = {}
for run in formal_runs:
    pred_counts = {config.CLASS_NAMES[i]: int(v) for i, v in enumerate(np.asarray(run["validation_metrics"]["confusion_matrix"]).sum(axis=0))}
    seed_history = run.get("history") or []
    seed_losses = [float(item["train_loss"]) for item in seed_history]
    # No Phase 4 smoke measurement in this notebook (already proven twice on real
    # hardware by the validation run) -- derive gradient/update evidence from the
    # real per-seed training trajectory instead of assuming it.
    training_progressed = len(seed_losses) >= 2 and seed_losses[-1] < seed_losses[0]
    gate_result = panderm_run.evaluate_non_collapse_gate(
        result=run, prediction_counts=pred_counts,
        backbone_gradient_verified=training_progressed, backbone_parameters_updated=training_progressed,
        identity_complete=True, no_test_access=True, provenance_allows_next_stage=True,
    )
    non_collapse_checks[f"seed_{run['seed']}"] = gate_result
formal_record = {
    "formal_status": "FORMAL TRAINING COMPLETED",
    "formal_training_allowed": True,
    "test_access_allowed": False,
    "run_version": RUN_VERSION,
    "git_commit": commit,
    "upstream_commit": upstream_commit,
    "checkpoint_sha256": checkpoint_sha256,
    "arch": ARCH,
    "variant": VARIANT,
    "seeds": list(SEEDS),
    "formal_epochs": FORMAL_EPOCHS,
    "warmup_epochs": WARMUP_EPOCHS,
    "aggregate_metrics": aggregate_output,
    "seed_runs": formal_runs,
    "non_collapse_gates": non_collapse_checks,
    "provenance": provenance_clearance,
    "claim_boundary": panderm_run.CLAIM_BOUNDARY,
    "evaluation_scope": "validation_only",
    "test_metrics": None,
}
panderm_run.write_json_atomic(
    FORMAL_ROOT / "formal_record.json", formal_record, write_guard=require_active_session
)
panderm_run.complete_sequential_session(ACTIVE_SESSION_PATH, session_id=FORMAL_SESSION_ID, history_directory=SESSION_HISTORY_DIR, checkpoint_integrity=formal_runs[0]["checkpoint_integrity"], result_identity={"epoch": formal_runs[0]["epoch"], "global_step": formal_runs[0]["global_step"], "run_identity_sha256": RUN_IDENTITY_SHA256})
print(json.dumps({"formal_status": "SUCCESS", "aggregate": aggregate_output}, indent=2))
